<a href="https://colab.research.google.com/github/maryamsohail32/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryamsohail32/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [22]:
import os, sys, subprocess

REPO_URL = "https://github.com/maryamsohail32/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if "google.colab" in sys.modules:
    os.chdir("/content")
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Ready.")

Working dir: /content/flyrank-ml-internship
Ready.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [23]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train rows:", len(train_df), "| Train clients:", train_df["client_id"].nunique())
print("Test rows:", len(test_df), "| Test clients:", test_df["client_id"].nunique())

overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print("Client overlap between train/test (should be 0):", len(overlap))

Train rows: 22885 | Train clients: 24
Test rows: 7115 | Test clients: 8
Client overlap between train/test (should be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [24]:
numeric_features = [
    "search_volume", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]

for col in numeric_features:
    train_df[f"has_{col}"] = train_df[col].notna().astype(int)
    test_df[f"has_{col}"] = test_df[col].notna().astype(int)

has_flags = [f"has_{col}" for col in numeric_features]

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
    ("flags", "passthrough", has_flags),
])

X_train = train_df[numeric_features + categorical_features + has_flags]
y_train = train_df["is_declining_label"]
X_test = test_df[numeric_features + categorical_features + has_flags]
y_test = test_df["is_declining_label"]

In [25]:
def visibility_score(d):
    return np.log1p(d["impressions_90d"].fillna(0))

def freshness_risk_score(d):
    return d["days_since_last_update"].fillna(0) / d["days_since_last_update"].max()

def position_opportunity_score(d):
    pos = d["avg_position"].replace(0, np.nan)  # 0 = no data, not rank 0
    return (1 / pos).fillna(0)

def depth_gap_score(d):
    return 1 - (d["word_count"].fillna(0) / (d["word_count"].max() + 1))

def baseline_score(d):
    v = visibility_score(d)
    f = freshness_risk_score(d)
    p = position_opportunity_score(d)
    g = depth_gap_score(d)
    def norm(x): return (x - x.min()) / (x.max() - x.min() + 1e-9)
    return 0.40*norm(v) + 0.30*norm(f) + 0.25*norm(p) + 0.05*norm(g)

test_df["baseline_score"] = baseline_score(test_df)

In [26]:
def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-scores)
    top_k_labels = np.array(y_true)[order][:k]
    return top_k_labels.mean()

base_rate = y_test.mean()
baseline_p50 = precision_at_k(y_test, test_df["baseline_score"], k=50)
print("Base rate (test set):", round(base_rate, 3))
print("Baseline rule Precision@50:", round(baseline_p50, 3))

Base rate (test set): 0.517
Baseline rule Precision@50: 0.28


In [27]:
from sklearn.linear_model import LogisticRegression

logreg_pipe = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])
logreg_pipe.fit(X_train, y_train)
logreg_scores = logreg_pipe.predict_proba(X_test)[:, 1]
logreg_p50 = precision_at_k(y_test, logreg_scores, k=50)
print("Logistic Regression Precision@50:", round(logreg_p50, 3))

Logistic Regression Precision@50: 0.7


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_pipe = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)),
])
rf_pipe.fit(X_train, y_train)
rf_scores = rf_pipe.predict_proba(X_test)[:, 1]
rf_p50 = precision_at_k(y_test, rf_scores, k=50)
print("Random Forest Precision@50:", round(rf_p50, 3))

In [ ]:
from sklearn.metrics import average_precision_score

results = pd.DataFrame({
    "method": ["base_rate (random)", "baseline rule", "logistic regression", "random forest"],
    "precision@50": [base_rate, baseline_p50, logreg_p50, rf_p50],
    "average_precision": [
        base_rate,
        average_precision_score(y_test, test_df["baseline_score"]),
        average_precision_score(y_test, logreg_scores),
        average_precision_score(y_test, rf_scores),
    ],
})
results.round(3)

**Reproducibility:** all models use `random_state=42`; split uses
`random_state=42`. Rerunning end to end should reproduce this table exactly.
Run `import sklearn; sklearn.__version__` if this number matters later —
tree-ensemble results can shift a couple points between library versions,
which is expected and worth one sentence of note, not alarm.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
feature_names = (
    numeric_features
    + list(rf_pipe.named_steps["prep"].named_transformers_["cat"]
           .named_steps["onehot"].get_feature_names_out(categorical_features))
    + has_flags
)
importances = rf_pipe.named_steps["clf"].feature_importances_
imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
imp_df.sort_values("importance", ascending=False).head(10)

In [ ]:
test_df["rf_score"] = rf_scores
wrong = test_df[(test_df["rf_score"] > 0.7) & (test_df["is_declining_label"] == 0)]
wrong[["content_id", "impressions_90d", "avg_position", "ctr",
       "days_since_last_update", "rf_score", "is_declining_label"]].head(3)

##  **Errors and interpretation:**

**Top features:** [fill in after running — name the actual top 3 from the
table above and say plainly why each plausibly relates to decline, e.g.
"days_since_last_update ranks high because staleness is a direct proxy for
untouched content — worth checking this isn't near-leakage from how
trend_direction gets computed upstream, since both trace back to time since
update."]

**Where the model is most wrong:** [fill in after inspecting the 3 wrong
cases above — describe what those pages have in common, e.g. "the model
over-trusts low avg_position pages even with thin impressions, suggesting it
hasn't fully learned that position only matters once there's enough traffic
behind it."]

**Sanity check on the top feature:** if one feature's importance looks
suspiciously dominant (e.g. carrying more than half of total importance),
that's a leakage flag, not a win — re-check it against the
`trend_direction`/`trend_pct` exclusion rule before trusting the model's
result.

**Bottom line vs baseline:** [state plainly, using your real numbers — did
RF beat the baseline at Precision@50? By how much? Is that gap big enough to
matter given it's still built on a proxy label, per ML-03's honest
limitation?]

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.